In [ ]:
# import matplotlib.pyplot as plt
import earthaccess

# from matplotlib.patches import Polygon
# from shapely.geometry import Polygon
from osgeo import gdal
import os
import numpy as np
import pandas as pd
from datetime import datetime
from datetime import timedelta
import shutil
from glob import glob
from ast import literal_eval
from pathlib import Path
import xarray as xr
import math
from scipy.ndimage import rotate
from pyresample import geometry, kd_tree

# from concurrent.futures import ThreadPoolExecutor
# import json

In [ ]:
# import matplotlib.pyplot as plt

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    MimeType,
    MosaickingOrder,
    SentinelHubRequest,
    bbox_to_dimensions,
)

In [ ]:
# Setting Account Info of Sentinel Hub
config = SHConfig()
config.sh_client_id = ""
config.sh_client_secret = ""
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

#### EMIT Data Download

In [ ]:
results = earthaccess.search_data(
    # native_id='EMIT_L1B_RAD_001_20230128T124118_2302809_033',
    short_name="EMITL1BRAD",
    # point=(87.865482,44.041006),
    temporal=("2023-10-15T14:09:21", "2023-10-15T14:09:21"),
    cloud_cover=(0, 100),
    count=100,
)
results[-1]

In [ ]:
merged_df = pd.read_csv("../s2_available_summary_2016_2024.csv")
# merged_df[merged_df['emission_auto'].isna()]['emission_auto']
merged_df_cleaned = merged_df.dropna(subset=["emission_auto"])
subset = merged_df_cleaned[
    merged_df_cleaned["provider"] == "NASA-JPL EMIT"
].copy()  # EMIT Dataset
temporal_list = (
    subset["datetime"].apply(lambda x: x.split("+")[0]).tolist()
)  # EMIT Data Time (285 data)

In [ ]:
# Download EMIT dataset
for temp in list(temporal_list):
    if temp == "2023-01-28T12:41:18":
        continue
    results = earthaccess.search_data(
        # native_id='EMIT_L1B_RAD_001_20230128T124118_2302809_033',
        short_name="EMITL1BRAD",
        # point=(-62.1123,-39.89402),
        temporal=(temp, temp),
        cloud_cover=(0, 100),
        count=100,
    )
    if len(results) == 0:
        print(temp, " is None")
        continue
    result = results[-1]
    earthaccess.download(result, "../EMIT_data/")

In [ ]:
EMIT_data = pd.DataFrame(
    columns=["plume_id", "plume_latitude", "plume_longitude", "time", "boundary"]
)
plume_ids, plume_lats, plume_longs, dts, boundaries = [], [], [], [], []
for k, v in subset.iterrows():
    plume_id, plume_lat, plume_long, dt = (
        v["plume_id"],
        v["plume_latitude"],
        v["plume_longitude"],
        v["datetime"],
    )
    results = earthaccess.search_data(
        short_name="EMITL1BRAD",
        point=(plume_long, plume_lat),
        temporal=(dt[:-3], dt[:-3]),
        cloud_cover=(0, 100),
        count=100,
    )
    res = results[-1]["umm"]
    print(res)
    gpoly = res["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]["GPolygons"][0][
        "Boundary"
    ]["Points"]
    coords = [(p["Longitude"], p["Latitude"]) for p in gpoly]
    # geom = Polygon(coords)
    # bbox = geom.bounds  # (minx, miny, maxx, maxy)
    # cx, cy = geom.centroid.x, geom.centroid.y
    plume_ids.append(plume_id)
    plume_lats.append(plume_lat)
    plume_longs.append(plume_long)
    dts.append(dt[:-3])
    boundaries.append(coords)
EMIT_data["plume_id"], EMIT_data["time"], EMIT_data["boundary"] = (
    plume_ids,
    dts,
    boundaries,
)
EMIT_data["plume_latitude"], EMIT_data["plume_longitude"] = plume_lats, plume_longs
# EMIT_data['bbox_minx'], EMIT_data['bbox_miny'], EMIT_data['bbox_maxx'], EMIT_data['bbox_maxy'] = bbox[0], bbox[1], bbox[2], bbox[3]
EMIT_data.to_csv("../EMIT_data.csv", index=False)

#### Sentinel-2 Data Download

In [ ]:
# Setting request image
# Return number of images
def zero_percentage(array):
    zero_count = np.count_nonzero(array == 0)
    zero_ratio = zero_count / array.size
    return zero_ratio


def download_img(
    betsiboka_coords_wgs84,
    betsiboka_bbox,
    betsiboka_size,
    resolution,
    time_interval,
    dir_name,
):
    cloud_coverage = (0, 10)
    evalscript_all_bands = """
        //VERSION=3
        
        function setup() {
            return {
                input: [{
                    bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B10","B11","B12"],
                    units: "DN"
                }],
                output: {
                    bands: 13,
                    sampleType: "INT16"
                }
            };
        }
    
        function evaluatePixel(sample) {
            return [sample.B01,
                    sample.B02,
                    sample.B03,
                    sample.B04,
                    sample.B05,
                    sample.B06,
                    sample.B07,
                    sample.B08,
                    sample.B8A,
                    sample.B09,
                    sample.B10,
                    sample.B11,
                    sample.B12];
        }
    """
    request_all_bands = SentinelHubRequest(
        data_folder=dir_name + "/",  # Path for saving images
        evalscript=evalscript_all_bands,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L1C.define_from(
                    "s2l1c", service_url=config.sh_base_url  # Sentinel-2 L1C Datset
                ),
                time_interval=time_interval,  # time period
                mosaicking_order=MosaickingOrder.LEAST_CC,  # least cloudy acquisitions
                # maxcc=0.1,
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],  # Output image format
        bbox=betsiboka_bbox,  # Interested area
        size=betsiboka_size,  # Image size
        config=config,
    )

    request_all_bands.custom_url_params = {
        "filter": {
            "timeRange": {"from": time_interval[0], "to": time_interval[1]},
            "cloudCoverage": cloud_coverage,
        }
    }

    # save image
    all_bands_response = request_all_bands.get_data(save_data=True)
    # print(all_bands_response)
    # print(type(all_bands_response))
    num = 0
    for single_response in all_bands_response:
        if zero_percentage(single_response[:, :, 0]) < 0.5:
            # single_response.save_data()
            num += 1
    return num

In [ ]:
non_list = [
    "2023-02-02",
    "2023-02-19",
    "2023-02-27",
    "2023-04-20",
    "2023-04-25",
    "2023-06-16",
    "2023-06-22",
    "2023-06-29",
    "2023-10-24",
    "2024-02-16",
    "2024-04-04",
    "2024-04-06",
    "2024-04-17",
    "2024-06-21",
]
full_non_list = [
    "2023-02-19T094118",
    "2023-02-27T172227",
    "2023-04-20T200240",
    "2023-04-25T051643",
    "2023-06-16T115949",
    "2023-06-22T115037",
    "2023-06-29T044014",
    "2023-06-29T044101",
    "2023-10-24T083639",
    "2024-02-16T105741",
    "2024-04-04T065751",
    "2024-04-06T083244",
    "2024-04-17T092338",
    "2024-06-21T191519",
]
for k, v in EMIT_data.iterrows():
    full_name = v["time"].replace(":", "")
    dir_name = "../S2_data/" + full_name
    date = v["time"][:10]
    if os.path.isdir(dir_name) or full_name in full_non_list:
        continue
    start_time = date + "T00:00:00"
    pts = literal_eval(v["boundary"])
    pts = np.array(list(map(list, pts)))
    # lon_i,lon_a,lat_i,lat_a = 94.04549407958984,95.57247161865234,26.723241424560547,28.048941040039062
    lon_i, lon_a, lat_i, lat_a = (
        pts[:, 0].min(),
        pts[:, 0].max(),
        pts[:, 1].min(),
        pts[:, 1].max(),
    )
    print(lon_i, lon_a, lat_i, lat_a)
    betsiboka_coords_wgs84 = (lon_i, lat_i, lon_a, lat_a)

    resolution = 60
    cloud_coverage = (0, 100)
    betsiboka_bbox = BBox(bbox=betsiboka_coords_wgs84, crs=CRS.WGS84)
    betsiboka_size = bbox_to_dimensions(betsiboka_bbox, resolution=resolution)
    print(f"Image shape at {resolution} m resolution: {betsiboka_size} pixels")

    time_obj = datetime.strptime(start_time, "%Y-%m-%dT%H:%M:%S")
    time_obj += timedelta(days=1)
    end_time = time_obj.strftime("%Y-%m-%dT%H:%M:%S")

    time_interval = (start_time, end_time)
    # os.mkdir('./'+dir_name)
    num_pic = download_img(
        betsiboka_coords_wgs84,
        betsiboka_bbox,
        betsiboka_size,
        resolution,
        time_interval,
        dir_name,
    )
    if num_pic == 0:
        shutil.rmtree("./" + dir_name)
        full_non_list.append(full_name)
    print(start_time, num_pic)
    # break

#### Dataset

##### Spatial Co-registration

In [ ]:
for k, v in EMIT_data.iterrows():
    # 0) S2 data
    s2_dir_name = v["time"].replace(":", "")
    # path = Path('../dataset/s2') / f'{s2_dir_name}.npy'
    # if s2_dir_name not in os.listdir('../S2_data/') or path.is_file():
    #     continue
    # print('Processing '+v['time'])
    if s2_dir_name not in os.listdir("../S2_data/") or k < 108:
        continue

    # 1) Read radiance ((lines, samples, bands))
    s2_tmp_name = s2_dir_name.replace("-", "")
    if s2_tmp_name == "20230403T111837":
        s2_tmp_name = "20230403T111825"
    if s2_tmp_name == "20230423T112832":
        s2_tmp_name = "20230423T112820"
    tag = f"EMIT_L1B_RAD_001_{s2_tmp_name}*.nc"
    in_path = sorted(Path("../EMIT_data").rglob(tag))
    ds = xr.open_dataset(in_path[0])
    rad = ds["radiance"].values  # (lines, samples, bands)
    rows, cols, bands = rad.shape

    # 2) Read RAD's location: latitude/longitude
    loc = xr.open_dataset(sorted(Path("../EMIT_data").rglob(tag))[0], group="location")
    loc_lat = (loc.get("latitude") or loc.get("lat")).values
    loc_lon = (loc.get("longitude") or loc.get("lon")).values

    assert loc_lat.shape == (rows, cols) and loc_lon.shape == (
        rows,
        cols,
    ), f"lat/lon Shape doesnt match radiance: lat={loc_lat.shape}, lon={loc_lon.shape}, rad={(rows,cols)}"

    # 3)Read S2 data
    img_data = gdal.Open(glob("../S2_data/" + s2_dir_name + "/**/*.tiff")[0])

    cols = img_data.RasterXSize
    rows = img_data.RasterYSize
    s2_data = np.zeros((rows, cols, 13), dtype=np.float32)
    for i in range(13):
        band = img_data.GetRasterBand(i + 1)
        s2_data[:, :, i] = band.ReadAsArray()

    # 4) Use same bbox + size Build S2
    gt = img_data.GetGeoTransform()
    x_min, px_w, _, y_max, _, px_h = gt
    lons_t = x_min + (np.arange(cols) + 0.5) * px_w
    lats_t = y_max + (np.arange(rows) + 0.5) * px_h
    grid_lon2d, grid_lat2d = np.meshgrid(lons_t, lats_t)  # Shape (height, width)

    EMIT_data = pd.read_csv("../EMIT_data.csv")
    pts = literal_eval(v["boundary"])
    pts = np.array(list(map(list, pts)))

    # 5) Calculate the rotation angle rotation north-up
    # ---- 1) principal direction, approximate the local tangent plane in meters (longitude multiplied by cosφ) ----
    lon = pts[:-1, 0]
    lat = pts[:-1, 1]
    lon0 = lon.mean()
    lat0 = lat.mean() * math.pi / 180.0
    x = (lon - lon0) * math.cos(
        lat0
    )  # The proportional constant for "meridian meters" can be omitted; the angle will not change.
    y = lat - lat.mean()
    X = np.vstack([x, y]).T
    C = np.cov(X.T)
    eigvals, eigvecs = np.linalg.eig(C)
    v = eigvecs[:, np.argmax(eigvals)]  # Principal axis direction (x, y)
    theta_east_ccw = math.degrees(
        math.atan2(v[1], v[0])
    )  # Counterclockwise angle (degrees) relative to "due east"
    theta_north_cw = (
        90 - theta_east_ccw
    )  # Convert to: Clockwise angle (degrees) relative to "due north"

    print("Main axis ∠ (counterclockwise relative to east):", theta_east_ccw)
    print("Main axis ∠ (clockwise relative to North):", theta_north_cw)

    # ---- 2) Angle for scipy.ndimage.rotate ----
    angle_to_vertical = +theta_north_cw
    angle_to_horizontal = -theta_east_ccw

    # 6) Rotate S2 to roughly align it with EMIT
    angle_deg = angle_to_vertical

    # Directly rotate the space in two dimensions, preserving the band dimension
    s2_data_rot = rotate(
        s2_data, angle=angle_to_vertical, axes=(0, 1), reshape=False, order=3
    )
    grid_lon2d_rot = rotate(
        grid_lon2d,
        angle=angle_to_vertical,
        axes=(0, 1),
        reshape=False,
        order=3,
    )  # mode="nearest")
    grid_lat2d_rot = rotate(
        grid_lat2d,
        angle=angle_to_vertical,
        axes=(0, 1),
        reshape=False,
        order=3,
    )  # mode="nearest")

    # 7) Precise matching defines the source (EMIT stripe) and the target (S2 grid)
    dst = geometry.SwathDefinition(lons=loc_lon, lats=loc_lat)
    src = geometry.GridDefinition(lons=grid_lon2d_rot, lats=grid_lat2d_rot)

    # 8) If the ROI (adjustable from 1000 to 3000 m) is too large, jagged edges will appear
    def resample_band_nearest(band2d, roi=500):
        return kd_tree.resample_nearest(
            src,
            band2d.astype(np.float32),
            dst,
            radius_of_influence=roi,
            fill_value=np.nan,
        )

    H, W, B = loc_lat.shape[0], loc_lat.shape[1], s2_data_rot.shape[2]

    # 9) Resample EMIT to the S2 grid (fully aligned with the saved S2 TIFF in terms of coverage/pixels).
    s2_on_rad = np.stack(
        [
            resample_band_nearest(s2_data_rot[:, :, b])
            for b in range(s2_data_rot.shape[2])
        ],
        axis=-1,
    )
    print(
        "S2 to EMIT Grid alignment complete:", s2_on_rad.shape
    )  # (height, width, bands)
    np.save("../dataset/s2/" + s2_dir_name + ".npy", s2_on_rad)

##### Patch

In [ ]:
# Remove clouds and low-quality images
# Remove black from images
# Split to generate paired datasets


def make_valid_mask(emit_cube, s2_cube, s2_nodata=0, min_emit_channels=1):
    """
    Generate pixel validity mask:
    - EMIT: Any >=min_emit_channels Each band is considered effective if it has a finite value
    - S2: Any band > s2_nodata (Default 0) is considered valid..
    """
    emit_valid = np.isfinite(emit_cube).sum(axis=2) >= min_emit_channels
    s2_valid = (np.isfinite(s2_cube) & (s2_cube > s2_nodata)).any(axis=2)
    return emit_valid & s2_valid


def tile_pairs(
    emit_cube,
    s2_cube,
    out_dir,
    src_id,
    tile=200,
    overlap=50,
    drop_black_frac=0.3,  # If the ratio of black/invalid pixels is > 0.3, discard.
    cloud_mask=None,
    cloud_frac_max=None,
    save_dtype=np.float32,
    compress=True,
):
    """
    Cut a set of paired large images into paired small pieces (200x200), which can overlap,
    remove large black/cloud areas, and output manifest.csv

    Parameter:
      emit_cube: (H,W,B_emit)
      s2_cube  : (H,W,B_s2) — Must be the same size and grid aligned with emit_cube
      out_dir  : Output directory
      src_id   : The unique identifier for this set of large images (written in the filename and manifest).
      tile     : Sub-image size (square)
      overlap  : Number of overlapping pixels (step size = tile - overlap)
      drop_black_frac: Threshold for the proportion of invalid/black pixels within a sub-image (if >), discard.
      cloud_mask: (H,W) bool, True indicates cloud; if provided, the cloud percentage will be calculated.
      cloud_frac_max: If the provided sub-map cloud ratio is greater than the threshold, it is discarded (e.g., 0.2).
      save_dtype: Save dtype
      compress   : Save .npz(Compressed) or .npy (Uncompressed)
    """
    H, W, Be = emit_cube.shape
    H2, W2, Bs = s2_cube.shape
    assert (H, W) == (H2, W2), "EMIT and S2 must be the same size!"

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    stride = tile - overlap
    assert stride > 0, "overlap must be smaller than tile"

    # Effective pixel mask
    valid_mask = make_valid_mask(emit_cube, s2_cube)

    records = []
    idx = 0

    # sliding windows
    for y in range(0, H - tile + 1, stride):
        for x in range(0, W - tile + 1, stride):
            vwin = valid_mask[y : y + tile, x : x + tile]
            black_frac = 1.0 - (vwin.sum() / (tile * tile))

            if black_frac > drop_black_frac:
                continue

            cloud_frac = None
            if cloud_mask is not None and cloud_frac_max is not None:
                cm_win = cloud_mask[y : y + tile, x : x + tile]
                cloud_frac = float(cm_win.sum() / (tile * tile))
                if cloud_frac > cloud_frac_max:
                    continue

            emit_tile = emit_cube[y : y + tile, x : x + tile, :].astype(
                save_dtype, copy=False
            )
            s2_tile = s2_cube[y : y + tile, x : x + tile, :].astype(
                save_dtype, copy=False
            )

            # Unified naming convention:{src}_{y}_{x}_ts{tile}_ov{overlap}.npz
            tile_id = f"{src_id}_y{y:05d}_x{x:05d}_ts{tile}_ov{overlap}"
            out_path = (
                out_dir / f"{tile_id}.npz" if compress else out_dir / f"{tile_id}.npy"
            )

            if compress:
                # One file
                np.savez_compressed(out_path, emit=emit_tile, s2=s2_tile)
            else:
                # Seperate file
                np.save(out_dir / f"{tile_id}_emit.npy", emit_tile)
                np.save(out_dir / f"{tile_id}_s2.npy", s2_tile)

            # records.append({
            #     "tile_id": tile_id,
            #     "src_id": src_id,
            #     "y": y, "x": x, "size": tile, "overlap": overlap,
            #     "emit_bands": Be,
            #     "s2_bands": Bs,
            #     "black_frac": float(black_frac),
            #     "cloud_frac": float(cloud_frac) if cloud_frac is not None else None,
            #     "path": str(out_path)
            # })
            idx += 1

    return records

In [ ]:
out_dir = "../dataset/tiles/"
records = []
dealed_id = []
for k, v in EMIT_data.iterrows():
    # 0) S2 data
    s2_dir_name = v["time"].replace(":", "")
    if s2_dir_name not in os.listdir("../S2_data/") or s2_dir_name in dealed_id:
        continue
    print("Processing " + v["time"])

    # 1) EMIT data radiance ((lines, samples, bands))
    s2_tmp_name = s2_dir_name.replace("-", "")
    if s2_tmp_name == "20230403T111837":
        s2_tmp_name = "20230403T111825"
    tag = f"EMIT_L1B_RAD_001_{s2_tmp_name}*.nc"
    in_path = sorted(Path("../EMIT_data").rglob(tag))
    ds = xr.open_dataset(in_path[0])
    emit_cube = ds["radiance"].values  # (lines, samples, bands)

    # 2) S2 data
    s2_cube = np.load("../dataset/s2/" + s2_dir_name + ".npy")

    # 3) patch
    src_id = s2_dir_name.replace("-", "")
    tile_pairs(
        emit_cube,
        s2_cube,
        out_dir,
        src_id,
        tile=500,
        overlap=100,  # tile=200, overlap=50,
        drop_black_frac=0.3,
        cloud_mask=None,
        cloud_frac_max=None,
        save_dtype=np.float32,
        compress=True,
    )
    # records.extend(res)
    dealed_id.append(s2_dir_name)

# df = pd.DataFrame.from_records(records)
# df_path = "../patch.csv"
# df.to_csv(df_path, index=False, encoding="utf-8")

In [ ]:
dataset = pd.DataFrame(columns=["id", "image_id", "y", "x", "cloudy"])
ids, image_ids, ys, xs = [], [], [], []
for file in os.listdir("../dataset/tiles_vis/"):
    ids.append(file.split(".")[0])
    img_id, yn, xn, _, _ = file.split("_")
    image_ids.append(img_id)
    ys.append(int(yn[1:]))
    xs.append(int(xn[1:]))
dataset["id"], dataset["image_id"], dataset["y"], dataset["x"] = ids, image_ids, ys, xs
dataset.to_csv("../dataset_new.csv", index=False)

##### Split

In [ ]:
from pathlib import Path
import random

random.seed(42)

tiles_dir = Path("../dataset/tiles/")
files = sorted(tiles_dir.glob("*.npz"))
n = len(files)

random.shuffle(files)

n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

train_files = files[:n_train]
val_files = files[n_train : n_train + n_val]
test_files = files[n_train + n_val :]

print(f"Total: {n}")
print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

out = Path("../dataset")
out.mkdir(parents=True, exist_ok=True)
(out / "train_new.txt").write_text("\n".join(str(p) for p in train_files))
(out / "val_new.txt").write_text("\n".join(str(p) for p in val_files))
(out / "test_new.txt").write_text("\n".join(str(p) for p in test_files))